## Electrostatic problem

In [15]:
from ngsolve import *
from netgen.read_gmsh import ReadGmsh
from ngsolve.webgui import Draw
from netgen.csg import *
import math
import pyvista as pv
import numpy as np
from ngsolve.krylovspace import GMRes

mesh_path = '../../meshes/coil_box_named'
output_path = '../../output/case1/case1_ngsolve'
output_path_gauss = '../../output/case1/gauss/case1_ngsolve'

# Import geometries
mesh = ReadGmsh(mesh_path + ".msh")

# for i in range(1, 3):
#     # print(i)
#     mesh.SetMaterial(i, f'{i}')

# for i in range(1, 13):
#     # print(i)
#     mesh.SetBCName(i-1, f'{i}')

mesh = Mesh(mesh)

mesh.ngmesh.Save(mesh_path + ".vol")

In [16]:
mesh.ne, mesh.nv, mesh.GetMaterials(), mesh.GetBoundaries()

(69629, 12090, ('vacuum', 'wire'), ('CoilIn', 'CoilOut'))

In [17]:
# Define material, coil, and BC parameters

I_coil = 2191.9 # Input electric current [A]
sigma_coil = 62.83185 # Coil's electric conductivity [S/m]

In [18]:
sigma = {"vacuum": 0.0, "wire": sigma_coil}  # Electric conductivity [S/m]
sigma_cf = CoefficientFunction([sigma.get(mat, 0.0) for mat in mesh.GetMaterials()])
crosssection = Integrate(1, mesh, definedon=mesh.Boundaries("CoilIn"))

print(f"Coil cross section = {crosssection} m^2.")

fespot = H1(mesh, order=1, definedon=mesh.Materials("wire"), dirichlet="CoilOut")
phi,psi = fespot.TnT()
with TaskManager():
    bfa = BilinearForm(sigma_cf*grad(phi)*grad(psi)*dx).Assemble()
    inv = bfa.mat.Inverse(freedofs=fespot.FreeDofs(), inverse="sparsecholesky")
    lff = LinearForm(I_coil/crosssection*psi*ds("CoilIn")).Assemble()
    gfphi = GridFunction(fespot)
    gfphi.vec.data = inv * lff.vec

gfcurrdens = -sigma_cf*grad(gfphi)

Coil cross section = 7.0710678118656e-05 m^2.


In [19]:
fespot_global = H1(mesh, order=1)
gfphi_global = GridFunction(fespot_global)
gfphi_global.Set(gfphi, definedon=mesh.Materials("wire"))

# fescurrden_global = VectorH1(mesh, order=1)
fescurrden_global = VectorL2(mesh, order=0)
# fescurrden_global = HCurl(mesh, order=1)
# fescurrden_global = HDiv(mesh, order=1)
gfcurrdens_global = GridFunction(fescurrden_global)
gfcurrdens_global.Set(gfcurrdens, definedon=mesh.Materials("wire"))

In [20]:
# vtk = VTKOutput(mesh,coefs=[gfphi],names=["sol"],filename=output_path + "electric_potential",subdivision=0)
# vtk.Do()

res = pv.read(mesh_path + ".msh")
points = res.points
gfphi_out = np.zeros((points.shape[0], 1))
gfcurrdens_out = np.zeros((points.shape[0], 3))

for i in range(points.shape[0]):
    point = mesh(points[i, 0], points[i, 1], points[i, 2])
    gfphi_out[i, :] = gfphi_global(point)
    gfcurrdens_out[i, :] = gfcurrdens_global(point)

res["electric_potential"] = gfphi_out
res["current_density"] = gfcurrdens_out

res.save(output_path + ".vtu")

In [21]:
from pprint import pprint
order_ir = 1 # Integration order
points_per_elem = 1 # Number of Gauss points per element
ir = IntegrationRule(ET.TET, order=order_ir)

gauss_coords = np.zeros((mesh.ne*points_per_elem, 3))
gauss_values1 = np.zeros((mesh.ne*points_per_elem, 1))
gauss_values2 = np.zeros((mesh.ne*points_per_elem, 3))

for i, el in enumerate(mesh.Elements(VOL)):
    # Get the transformation for the element
    trafo = mesh.GetTrafo(el)
    # Coordinates
    # coords = trafo(ir)
    # print(coords[0])
    # Values corresponding to the coordinates
    # vals = A(trafo(ir))


    for j, ip in enumerate(ir):
        # Coordinate
        coord = trafo(ip)
        # Values corresponding to the coordinate
        val1 = gfphi_global(trafo(ip))
        val2 = gfcurrdens_global(trafo(ip))

        gauss_coords[i+j, :] = coord.point
        gauss_values1[i+j, :] = val1
        gauss_values2[i+j, :] = val2

    # if i == 60:
    #    pprint(dir(coord))
    #    print(coord.point)

results = np.concatenate((gauss_coords, gauss_values1, gauss_values2), axis=1)

print(results.shape)

print("Min and max X coord:")
print(np.min(gauss_coords[:, 0]))
print(np.max(gauss_coords[:, 0]))

print("Min and max Y coord:")
print(np.min(gauss_coords[:, 1]))
print(np.max(gauss_coords[:, 1]))

print("Min and max Z coord:")
print(np.min(gauss_coords[:, 2]))
print(np.max(gauss_coords[:, 2]))

np.save(output_path_gauss + ".npy", results)

# np.save("coords.npy", gauss_coords)
# np.save("vals.npy", gauss_values)

(69629, 7)
Min and max X coord:
-0.09800790594122501
0.0979479051960425
Min and max Y coord:
-0.099551605255125
0.09795298646876001
Min and max Z coord:
-0.09789693344965
0.09789017226766
